%%<br>
Web scraping from Wikipedia

In [1]:
wiki_url = 'https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

%%<br>
import required packages

In [3]:
import numpy as np, datetime as dt
import requests
from bs4 import BeautifulSoup
import re, unicodedata, pandas as pd
# %%
#  Helper function
def save_to_dict(launch_dict, row_id, elems):

    # Date value
    date_time = elems[columns_name.index("Date and time ( UTC )")-1]
    date_time = ' '.join(list(date_time.strings)[:2])
    date_time = dt.datetime.strptime(str(date_time), "%B %d, %Y %H:%M")
    launch_dict['Date and time ( UTC )'].append(date_time)
    launch_dict['Date'].append(date_time.date())
    launch_dict['Time'].append(date_time.time())

    # Booster version
    booster_version = elems[columns_name.index("Version, booster")-1]
    try:
        booster_version = ' '.join(booster_version.strings)
        booster_version = re.sub('[\xa0] ', '-', booster_version).replace(' ‑ ', '-')
    except:
        booster_version = None
        
    launch_dict['Version, booster'].append(booster_version)
    launch_dict['Version Booster'].append(booster_version)

    # Launch Site
    site = elems[columns_name.index('Launch site')-1]
    if site.strings:
        launch_site = ''.join(site.strings)
    else:
        launch_site = None
    launch_dict['Launch site'].append(launch_site)

    # Payload
    payload = elems[columns_name.index('Payload')-1]
    if payload.strings:
        payload = ''.join(payload.strings)
    else:
        payload = None
    launch_dict['Payload'].append(payload)

    # Payload Mass
    payload_mass = elems[columns_name.index('Payload mass')-1]
    if payload_mass.strings:
        payload_mass = ''.join(payload_mass.strings)
        payload_mass = re.sub('[\xa0]', ' ', payload_mass)
        payload_mass = payload_mass[:payload_mass.find('kg')+2]
    else:
        payload_mass = None
    launch_dict['Payload mass'].append(payload_mass)
    
    # Orbit
    orbit = elems[columns_name.index('Orbit')-1]
    if orbit.strings:
        orbit = ''.join(orbit.strings)
    else:
        orbit = None
    launch_dict['Orbit'].append(orbit)

    # Customer
    customer = elems[columns_name.index('Customer')-1]
    if customer.strings:
        customer = ''.join(customer.strings)
    else:
        customer = None
    launch_dict['Customer'].append(customer)

    # Launch outcome
    launch_outcome = elems[columns_name.index('Launch outcome')-1]
    if launch_outcome.strings:
        launch_outcome = ''.join(launch_outcome.strings)
    else:
        launch_outcome = None
    launch_dict['Launch outcome'].append(launch_outcome)

    # Booster landing
    booster_landing = elems[columns_name.index('Booster landing')-1]
    if booster_landing.strings:
        booster_landing = ''.join(booster_landing.strings)
    else:
        booster_landing = None
    launch_dict['Booster landing'].append(booster_landing)

In [4]:
def extract_column_from_header(row):
    """
    Return the landing status from the input HTML table cell
    """
    if row.br:
        row.br.replace_with(' ')
    for a_tag in row.find_all('a'):
        a_tag.replace_with(a_tag.get_text())
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.stripped_strings)

    # Filter the digit and empty names
    if not (column_name.strip().isdigit()):
        return column_name.strip()

%%<br>
Request the Falcon9 Launch Wiki HTML

In [5]:
response = requests.get(wiki_url, headers=headers)
print('status code:', response.status_code)

status code: 200


%%<br>
Create a beautifulsoup object

In [6]:
soup = BeautifulSoup(response.text, features="html.parser", )
print(soup.title)
# %%
# Extract all columns/variable names from the HTML
html_tables = soup.find_all('table')

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


using the third table which is the first launch table

In [14]:
first_launch_table = html_tables[2]
print(first_launch_table)

<table class="wikitable plainrowheaders collapsible sticky-header" id="2025ytd" style="width: 100%;">
<tbody id="mwAUU"><tr id="mwAUY"><th id="mwAUc" scope="col">Flight No.</th>
<th id="mwAUg" scope="col">Date and time (UTC)</th>
<th id="mwAUs" scope="col">Version, booster</th>
<th id="mwAVI" scope="col">Launch site</th>
<th id="mwAVQ" scope="col">Payload</th>
<th id="mwAVk" scope="col">Payload mass</th>
<th id="mwAVo" scope="col">Orbit</th>
<th id="mwAVs" scope="col">Customer</th>
<th id="mwAVw" scope="col">Launch outcome</th>
<th id="mwAV4" scope="col">Booster landing</th></tr>
<tr id="F9-418">
<th id="mwAWE" rowspan="2" scope="row" style="text-align:center;">418</th>
<td id="mwAWI"><span about="#mwt180" data-mw='{"parts":[{"template":{"target":{"wt":"date table sorting","href":"./Template:Date_table_sorting"},"params":{"1":{"wt":"January 4, 2025"}},"i":0}}]}' data-sort-value="000000002025-01-04-0000" id="mwAWM" style="white-space:nowrap" typeof="mw:Transclusion">January 4, 2025</spa

In [8]:
columns_name = []
for th in first_launch_table.find_all('tr')[0].find_all('th'): # all headers in first row
    columns_name.append(extract_column_from_header(th))

In [9]:
print(columns_name)
# %%
# Create dataframe from HTML table
launch_dict = {k: [] for k in columns_name}

['Flight No.', 'Date and time ( UTC )', 'Version, booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


Add some new columns

In [10]:
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

Parsing into dict<br>
Extract each table of class "wikitable plainrowheaders collapsible sticky-header"

In [11]:
for table_number, table in enumerate(soup.find_all('table',
    attrs={'class':"wikitable plainrowheaders collapsible sticky-header"}) ):
    # get table row
    print('\ntable', table_number)
    for row in table.find_all("tr"):
        # check if it is a row
        if row.th and row.th['scope']=='row':
            # get number for Flight No.
            flight_number = row.th.string.strip()
            try:
                row_id = int(flight_number)
            except:
                row_id = str(flight_number)

            # print(flight_number)
            # get row elements
            elems = row.find_all('td')
            # if the row's elements match the number of columns ( minus "Flight No.")
            if len(elems) == 9:                
                save_to_dict(launch_dict, row_id, elems)
                # Flight Number 
                launch_dict['Flight No.'].append(row_id)
                # print(launch_dict['Flight No.'])


table 0

table 1


%%<br>
Creating data frame from dictionary

In [12]:
df = pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

In [16]:
## Checking Data frame
df.head()

,Flight No.,Date and time ( UTC ),"Version, booster",Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing,Version Booster,Date,Time
0,418,2025-01-04 01:27:00,F9 -B5 B1073-20,"Cape Canaveral, SLC‑40",Thuraya 4-NGS,"5,000 kg",GTO,Thuraya,Success,Success (ASOG),F9 -B5 B1073-20,2025-01-04,01:27:00
1,419,2025-01-06 20:43:00,F9 -B5 B1077-17,"Cape Canaveral, SLC‑40",Starlink: Group 6‑71,"~17,500 kg",LEO,SpaceX,Success,Success (JRTI),F9 -B5 B1077-17,2025-01-06,20:43:00
2,420,2025-01-08 15:27:00,F9 -B5 B1086-3,"Kennedy, LC‑39A",Starlink: Group 12-11 (21 satellites),"~16,500 kg",LEO,SpaceX,Success,Success (ASOG),F9 -B5 B1086-3,2025-01-08,15:27:00
3,421,2025-01-10 03:53:00,F9 -B5 B1071-22,"Vandenberg, SLC‑4E",NROL-153 (22 Starshield satellites)[34],U,LEO,NRO,Success,Success (OCISLY),F9 -B5 B1071-22,2025-01-10,03:53:00
4,422,2025-01-10 19:11:00,F9 -B5 B1067-25,"Cape Canaveral, SLC‑40",Starlink: Group 12-12 (21 satellites),"~16,500 kg",LEO,SpaceX,Success,Success (JRTI),F9 -B5 B1067-25,2025-01-10,19:11:00


In [17]:
df.sample(5)

,Flight No.,Date and time ( UTC ),"Version, booster",Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing,Version Booster,Date,Time
107,525,2025-08-31 11:49:00,F9 -B5 B1077-23,"Cape Canaveral, SLC‑40",Starlink: Group 10-14,"~16,100 kg",LEO,SpaceX,Success,Success (JRTI),F9 -B5 B1077-23,2025-08-31,11:49:00
39,457,2025-04-07 23:06:00,F9 -B5 B1093-1,"Vandenberg, SLC‑4E",Starlink: Group 11-11,"~15,500 kg",LEO,SpaceX,Success,Success (OCISLY),F9 -B5 B1093-1,2025-04-07,23:06:00
10,428,2025-01-24 14:07:00,F9 -B5 B1063-23,"Vandenberg, SLC‑4E",Starlink: Group 11-6 (23 satellites),"~17,100 kg",LEO,SpaceX,Success,Success (OCISLY),F9 -B5 B1063-23,2025-01-24,14:07:00
20,438,2025-02-18 23:21:00,F9 -B5 B1080-16,"Cape Canaveral, SLC‑40",Starlink: Group 10-12 (23 satellites),"~17,100 kg",LEO,SpaceX,Success,Success (JRTI),F9 -B5 B1080-16,2025-02-18,23:21:00
211,629,2026-04-19 16:03:00,F9 -B5 B1097-8,"Vandenberg, SLC‑4E",Starlink: Group 17-22,"~14,375 kg",LEO,SpaceX,Success,Success (OCISLY),F9 -B5 B1097-8,2026-04-19,16:03:00


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Flight No.             245 non-null    object        
 1   Date and time ( UTC )  245 non-null    datetime64[us]
 2   Version, booster       245 non-null    str           
 3   Launch site            245 non-null    str           
 4   Payload                245 non-null    str           
 5   Payload mass           245 non-null    str           
 6   Orbit                  245 non-null    str           
 7   Customer               245 non-null    str           
 8   Launch outcome         245 non-null    str           
 9   Booster landing        245 non-null    str           
 10  Version Booster        245 non-null    str           
 11  Date                   245 non-null    object        
 12  Time                   245 non-null    object        
dtypes: datetime64[us

In [19]:
df.describe(include='all')

,Flight No.,Date and time ( UTC ),"Version, booster",Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing,Version Booster,Date,Time
count,245.0,245,245,245,245,245,245,245,245,245,245,245,245
unique,245.0,NaN,245,5,245,45,9,32,1,9,245,220,226
top,418.0,NaN,F9 -B5 B1073-20,"Cape Canaveral, SLC‑40",Thuraya 4-NGS,"~16,675 kg",LEO,SpaceX,Success,Success (OCISLY),F9 -B5 B1073-20,2025-01-10,02:09:00
freq,1.0,NaN,1,111,1,43,158,185,245,93,1,2,3
mean,NaN,2025-10-03 02:32:59.755102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,2025-01-04 01:27:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,2025-05-24 17:19:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,2025-09-26 04:26:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,2026-02-16 07:59:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,2026-07-07 07:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


%%<br>
Save to csv

In [20]:
df.to_csv('data/spacex_web_scraped.csv', index=False)
# %%